# MetroRide Phase 2: add live driver-app events

Phase 1 left us a lakehouse with the April–May trips in `bronze.trips_raw` and `silver.trips`. In phase 2 MetroRide also receives **driver-app events** (`accepted`, `gps_ping`, `arrived_at_pickup`, `cancelled`). They tell us when the driver actually reached the pickup point, so the gold table can split the passenger's wait into two measures: the driver's trip to the pickup point, and the time until the passenger was picked up.

```
trips_YYYY-MM.csv ─────────▶ bronze.trips_raw ─────────▶ silver.trips ─────────┐
                                                                               ├─▶ silver.trip_context ─▶ gold.daily_service_performance
driver_events_YYYY-MM.csv ─▶ bronze.driver_events_raw ─▶ silver.driver_events ─┘
```

| Step | What you do |
|---|---|
| 1 | Create the additional tables |
| 2 | Alter existing tables |
| 3 | Ingest June data into bronze |
| 4 | Transform June: bronze → silver |
| 5 | Transform June: silver → gold |
| 6–8 | The same three steps for July |
| 9–11 | The same three steps for August |
| 12 | Display data in the bronze, silver and gold layers |
| 13 | Schema evolution: look closely at the evolved gold table |

**What changes in phase 2**

| Table | Phase 1 | Phase 2 |
|---|---|---|
| `bronze.trips_raw` | exists | unchanged (the trip CSV has the same columns) |
| `bronze.driver_events_raw` | – | **new** |
| `silver.trips` | exists | unchanged |
| `silver.driver_events` | – | **new** |
| `silver.trip_context` | – | **new**: trips joined with the driver's arrival |
| `gold.daily_service_performance` | a "try it yourself" exercise | created with the phase-1 columns, then **two columns are added** |

**Before you start**

* Run the phase-1 tutorial (`phases/phase-1-batch/tutorial.ipynb`) first. This notebook starts from a **copy** of its lakehouse, so phase 1 is left untouched.
* Run this notebook from the `phases/phase-2-streaming/` folder, with the same packages as phase 1: `pip install "pyducklake[pandas]" pytz`.

## 0. Setup: start from a copy of the phase-1 lakehouse

We copy `../phase-1-batch/lakehouse/` to `lakehouse/` in this folder and continue there.

One detail needs care. A DuckLake catalog remembers the **absolute path of its data folder** (`data_path`); the paths of the individual Parquet files are stored relative to it. After the copy, the catalog still points to the phase-1 folder, and pyducklake refuses to open it with a different `data_path`. Because the catalog is an ordinary DuckDB database, we fix this with one SQL `UPDATE` on its `ducklake_metadata` table before we open it with pyducklake.

In [1]:
import hashlib
import os
import shutil
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pacsv
from pyducklake import (
    Catalog, Schema, required, optional,
    StringType, IntegerType, TimestampTZType, DecimalType, DateType,
)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

BKK = ZoneInfo("Asia/Bangkok")
AUTHOR = "phase-2 tutorial"

PHASE_DIR = Path.cwd()                                          # .../phases/phase-2-streaming
SOURCE_DIR = PHASE_DIR / "bronze"                               # trip and driver-event CSVs for June-August
LAKE_DIR = PHASE_DIR / "lakehouse"                              # the phase-2 lakehouse
PHASE1_LAKE_DIR = PHASE_DIR.parent / "phase-1-batch" / "lakehouse"
assert SOURCE_DIR.exists(), "Run this notebook from the phases/phase-2-streaming/ folder"
assert (PHASE1_LAKE_DIR / "metadata.ducklake").exists(), "Run the phase-1 tutorial first"

# Start from a fresh copy of the phase-1 lakehouse every time the tutorial runs
if LAKE_DIR.exists():
    shutil.rmtree(LAKE_DIR)
shutil.copytree(PHASE1_LAKE_DIR, LAKE_DIR, ignore=shutil.ignore_patterns("landing"))

# Point the copied catalog at the copied data folder
new_data_path = str(LAKE_DIR / "data") + "/"
with duckdb.connect(str(LAKE_DIR / "metadata.ducklake")) as meta:
    old_data_path = meta.sql("SELECT value FROM ducklake_metadata WHERE key = 'data_path'").fetchone()[0]
    meta.execute("UPDATE ducklake_metadata SET value = ? WHERE key = 'data_path'", [new_data_path])
# paths are printed relative to the phases/ folder
print("data_path before:", os.path.relpath(old_data_path, PHASE_DIR.parent))
print("data_path after :", os.path.relpath(new_data_path, PHASE_DIR.parent))

catalog = Catalog(
    "metroride",
    str(LAKE_DIR / "metadata.ducklake"),
    data_path=str(LAKE_DIR / "data"),
)
catalog.set_option("data_inlining_row_limit", "0")   # always write Parquet files, as in phase 1

con = catalog.connection
con.execute("SET TimeZone = 'Asia/Bangkok'")


def sql(query: str) -> pd.DataFrame:
    # Run a SQL query on the lakehouse and return a pandas DataFrame.
    return con.sql(query).df()


def schema_df(table) -> pd.DataFrame:
    # Show a pyducklake table schema as a small DataFrame.
    return pd.DataFrame(
        [(f.name, str(f.field_type), "required" if f.required else "optional") for f in table.schema.fields],
        columns=["column", "type", "nullability"],
    )


def snapshots_df() -> pd.DataFrame:
    # All snapshots of the catalog (snapshots are catalog-wide).
    df = catalog.load_table(("bronze", "trips_raw")).inspect().snapshots().to_pandas()
    return df[["snapshot_id", "snapshot_time", "schema_version", "changes", "author", "commit_message"]]


PHASE1_LAST_SNAPSHOT = int(sql("SELECT max(snapshot_id) AS id FROM metroride.snapshots()").id[0])
print("Last snapshot written by phase 1:", PHASE1_LAST_SNAPSHOT)

data_path before: phase-1-batch/lakehouse/data
data_path after : phase-2-streaming/lakehouse/data
Last snapshot written by phase 1: 8


What phase 1 left us:

In [2]:
for ns in catalog.list_namespaces():
    print(f"{ns:8s} -> {catalog.list_tables(ns)}")

sql('''
    SELECT 'bronze.trips_raw' AS table_name, count(*) AS rows FROM metroride.bronze.trips_raw
    UNION ALL
    SELECT 'silver.trips', count(*) FROM metroride.silver.trips
''')

bronze   -> [('bronze', 'trips_raw')]
main     -> []
silver   -> [('silver', 'trips')]


,table_name,rows
0,bronze.trips_raw,269
1,silver.trips,269


## 1. Create the additional tables

| Table | Grain | Notes |
|---|---|---|
| `bronze.driver_events_raw` | one **received** event row | All source columns as text, plus the same four ingest-metadata columns as `bronze.trips_raw`. A repeated delivery is stored twice. |
| `silver.driver_events` | one row per `event_id` | Typed: `event_at` is a timestamp with time zone, coordinates are `decimal(9,5)`. |
| `silver.trip_context` | one row per `trip_id` | Every column of `silver.trips`, plus `driver_arrived_at`, `driver_arrival_wait_minutes` and `passenger_pickup_wait_minutes`. |
| `gold.daily_service_performance` | one row per `service_date` and `time_slot` | Created with the **phase-1 columns**. Step 2 adds the phase-2 columns. |

### 1.1 Bronze

In [3]:
TRIP_COLUMNS = [
    "trip_id", "driver_id", "service_zone", "requested_at", "accepted_at",
    "picked_up_at", "completed_at", "cancelled_at", "status", "fare_thb",
]
EVENT_COLUMNS = ["event_id", "trip_id", "driver_id", "event_type", "event_at", "latitude", "longitude"]


def ingest_metadata_columns():
    # The four ingest-metadata columns that every bronze table carries (same as phase 1).
    return [
        required("source_file", StringType()),
        required("source_row_number", IntegerType()),
        required("file_sha256", StringType()),
        required("ingested_at", TimestampTZType()),
    ]


bronze_trips = catalog.load_table(("bronze", "trips_raw"))          # exists since phase 1
bronze_events = catalog.create_table(
    ("bronze", "driver_events_raw"),
    Schema.of(
        *[optional(col, StringType()) for col in EVENT_COLUMNS],     # raw source columns, all text
        *ingest_metadata_columns(),
    ),
)
schema_df(bronze_events)

,column,type,nullability
0,event_id,StringType(),optional
1,trip_id,StringType(),optional
2,driver_id,StringType(),optional
3,event_type,StringType(),optional
4,event_at,StringType(),optional
5,latitude,StringType(),optional
6,longitude,StringType(),optional
7,source_file,StringType(),required
8,source_row_number,IntegerType(),required
9,file_sha256,StringType(),required


### 1.2 Silver

`silver.trip_context` reuses the field list of `silver.trips`, so the two tables cannot drift apart.

In [4]:
silver_trips = catalog.load_table(("silver", "trips"))              # exists since phase 1

silver_events = catalog.create_table(
    ("silver", "driver_events"),
    Schema.of(
        required("event_id", StringType()),
        required("trip_id", StringType()),
        required("driver_id", StringType()),
        required("event_type", StringType()),
        required("event_at", TimestampTZType()),
        required("latitude", DecimalType(9, 5)),
        required("longitude", DecimalType(9, 5)),
    ),
)

trip_context = catalog.create_table(
    ("silver", "trip_context"),
    Schema.of(
        *silver_trips.schema.fields,                                 # every column of silver.trips
        optional("driver_arrived_at", TimestampTZType()),
        optional("driver_arrival_wait_minutes", DecimalType(8, 2)),
        optional("passenger_pickup_wait_minutes", DecimalType(8, 2)),
    ),
)
schema_df(trip_context)

,column,type,nullability
0,trip_id,StringType(),required
1,driver_id,StringType(),required
2,service_zone,StringType(),required
3,status,StringType(),required
4,requested_at,TimestampTZType(),required
5,accepted_at,TimestampTZType(),required
6,picked_up_at,TimestampTZType(),optional
7,completed_at,TimestampTZType(),optional
8,cancelled_at,TimestampTZType(),optional
9,fare_thb,"DecimalType(precision=10, scale=2)",optional


### 1.3 Gold, as phase 1 would have built it

Phase 1 left the gold table as an exercise. We create it now **with the phase-1 columns only** and fill it with April and May from `silver.trips`. That way gold already holds data written with its old schema when Step 2 changes the schema.

| Column | Calculation |
|---|---|
| `requests`, `completed_trips`, `cancelled_trips` | count all, completed and cancelled trips |
| `cancellation_rate_pct` | `100 × cancelled_trips / requests` |
| `gross_fare_thb` | sum of fares of completed trips |
| `average_passenger_pickup_wait_minutes` | average `picked_up_at − requested_at` of completed trips |

In [5]:
catalog.create_namespace("gold")
gold = catalog.create_table(
    ("gold", "daily_service_performance"),
    Schema.of(
        required("service_date", DateType()),
        required("time_slot", StringType()),
        required("requests", IntegerType()),
        required("completed_trips", IntegerType()),
        required("cancelled_trips", IntegerType()),
        required("cancellation_rate_pct", DecimalType(5, 2)),
        required("gross_fare_thb", DecimalType(12, 2)),
        optional("average_passenger_pickup_wait_minutes", DecimalType(8, 2)),
    ),
)

GOLD_PHASE1_SQL = '''
SELECT
    service_date,
    time_slot,
    CAST(count(*) AS INTEGER)                                AS requests,
    CAST(count(*) FILTER (status = 'completed') AS INTEGER)  AS completed_trips,
    CAST(count(*) FILTER (status = 'cancelled') AS INTEGER)  AS cancelled_trips,
    CAST(round(100.0 * count(*) FILTER (status = 'cancelled') / count(*), 2) AS DECIMAL(5, 2))
                                                             AS cancellation_rate_pct,
    CAST(coalesce(sum(fare_thb) FILTER (status = 'completed'), 0) AS DECIMAL(12, 2))
                                                             AS gross_fare_thb,
    CAST(round(avg(date_diff('second', requested_at, picked_up_at) / 60.0)
               FILTER (status = 'completed'), 2) AS DECIMAL(8, 2))
                                                             AS average_passenger_pickup_wait_minutes
FROM metroride.silver.trips
GROUP BY service_date, time_slot
ORDER BY service_date, time_slot DESC
'''

with catalog.begin_transaction() as txn:
    catalog.set_commit_message("gold: build April-May with the phase-1 columns", author=AUTHOR)
    txn.load_table(("gold", "daily_service_performance")).append(con.sql(GOLD_PHASE1_SQL).to_arrow_table())

print("Rows in gold.daily_service_performance:", gold.scan().count())
gold.scan().with_limit(4).to_pandas()

Rows in gold.daily_service_performance: 122


,service_date,time_slot,requests,completed_trips,cancelled_trips,cancellation_rate_pct,gross_fare_thb,average_passenger_pickup_wait_minutes
0,2026-04-01,morning,2,2,0,0.0,309.0,9.50
1,2026-04-01,evening,3,3,0,0.0,505.0,13.33
2,2026-04-02,morning,2,2,0,0.0,375.0,7.50
3,2026-04-02,evening,2,2,0,0.0,379.0,10.50


In [6]:
for ns in catalog.list_namespaces():
    print(f"{ns:8s} -> {catalog.list_tables(ns)}")

bronze   -> [('bronze', 'driver_events_raw'), ('bronze', 'trips_raw')]
gold     -> [('gold', 'daily_service_performance')]
main     -> []
silver   -> [('silver', 'driver_events'), ('silver', 'trip_context'), ('silver', 'trips')]


## 2. Alter existing tables

### 2.1 `bronze.trips_raw` and `silver.trips`: no change needed

The phase-2 trip CSVs have exactly the same columns as in phase 1, so both trip tables stay as they are. We check the headers instead of assuming it:

In [7]:
for csv_file in sorted(SOURCE_DIR.glob("trips_*.csv")):
    header = csv_file.read_text().splitlines()[0].split(",")
    print(f"{csv_file.name}: {'same columns as phase 1' if header == TRIP_COLUMNS else 'CHANGED: ' + str(header)}")

trips_2026-06.csv: same columns as phase 1
trips_2026-07.csv: same columns as phase 1
trips_2026-08.csv: same columns as phase 1


### 2.2 `gold.daily_service_performance`: add two columns

The phase-2 spec keeps the gold grain (one row per date and slot) and adds two measures:

| New column | Type | Calculation |
|---|---|---|
| `average_driver_arrival_wait_minutes` | decimal(8,2), nullable | average request-to-driver-arrival minutes of completed trips |
| `arrived_within_8_minutes_pct` | decimal(5,2), nullable | `100 × completed trips with arrival wait ≤ 8 / completed trips with an arrival event` |

`table.update_schema()` collects schema changes and commits them when the `with` block ends. We run it inside a transaction so the snapshot carries a commit message.

In DuckLake, adding a column is a **metadata-only** change: the catalog records the new columns in a new snapshot, and the existing Parquet files are **not rewritten**. Rows already in the table read as `NULL` in the new columns. Both columns must therefore be nullable.

In [8]:
gold_snapshot_before_alter = gold.current_snapshot().snapshot_id

with catalog.begin_transaction() as txn:
    catalog.set_commit_message("gold: add driver-arrival measures", author=AUTHOR)
    with txn.load_table(("gold", "daily_service_performance")).update_schema() as update:
        update.add_column("average_driver_arrival_wait_minutes", DecimalType(8, 2))
        update.add_column("arrived_within_8_minutes_pct", DecimalType(5, 2))

gold = gold.refresh()                       # reload the table object so it sees the new schema
gold_snapshot_after_alter = gold.current_snapshot().snapshot_id
print(f"Gold schema changed between snapshot {gold_snapshot_before_alter} and {gold_snapshot_after_alter}")
schema_df(gold)

Gold schema changed between snapshot 14 and 15


,column,type,nullability
0,service_date,DateType(),required
1,time_slot,StringType(),required
2,requests,IntegerType(),required
3,completed_trips,IntegerType(),required
4,cancelled_trips,IntegerType(),required
5,cancellation_rate_pct,"DecimalType(precision=5, scale=2)",required
6,gross_fare_thb,"DecimalType(precision=12, scale=2)",required
7,average_passenger_pickup_wait_minutes,"DecimalType(precision=8, scale=2)",optional
8,average_driver_arrival_wait_minutes,"DecimalType(precision=8, scale=2)",optional
9,arrived_within_8_minutes_pct,"DecimalType(precision=5, scale=2)",optional


The April–May rows are still there. They have no values for the new columns yet:

In [9]:
gold.scan().with_limit(4).to_pandas()

,service_date,time_slot,requests,completed_trips,cancelled_trips,cancellation_rate_pct,gross_fare_thb,average_passenger_pickup_wait_minutes,average_driver_arrival_wait_minutes,arrived_within_8_minutes_pct
0,2026-04-01,morning,2,2,0,0.0,309.0,9.50,NaN,NaN
1,2026-04-01,evening,3,3,0,0.0,505.0,13.33,NaN,NaN
2,2026-04-02,morning,2,2,0,0.0,375.0,7.50,NaN,NaN
3,2026-04-02,evening,2,2,0,0.0,379.0,10.50,NaN,NaN


## 3. Ingest June data into the bronze layer

`ingest_csv()` generalises the phase-1 ingest function to any bronze table:

1. **File guard** (all tables): a file whose SHA-256 hash is already in the table is skipped.
2. **Row guard** (only when a `key` is given): rows whose key is already in bronze, or repeated inside the file, are skipped. We use it for trips (`key="trip_id"`).

Driver events get **no row guard**. Bronze is a log of what was *received*, and the phase-2 spec says a repeated delivery of the same event is kept in bronze. Removing it is silver's job.

`ingest_month()` loads both files of one month: trips first, then driver events.

In [10]:
def ingest_csv(csv_path: Path, table_name: str, columns: list[str], key: str | None = None) -> int:
    # Append one CSV to bronze.<table_name>. Returns the number of rows inserted.
    # key: column that identifies a record (row guard); None keeps every received row.
    table = catalog.load_table(("bronze", table_name))
    file_hash = hashlib.sha256(csv_path.read_bytes()).hexdigest()

    # Guard 1: the exact same file was ingested before
    if table.scan(f"file_sha256 = '{file_hash}'").count() > 0:
        print(f"SKIP {csv_path.name}: this file is already in bronze.{table_name}")
        return 0

    # Read every column as text; blank cells become NULL
    raw = pacsv.read_csv(
        csv_path,
        convert_options=pacsv.ConvertOptions(
            column_types={col: pa.string() for col in columns},
            strings_can_be_null=True,
        ),
    )
    if raw.column_names != columns:
        raise ValueError(f"Unexpected header in {csv_path.name}: {raw.column_names}")

    # Add ingest metadata columns
    n = raw.num_rows
    batch = (
        raw.append_column("source_file", pa.array([csv_path.name] * n, pa.string()))
           .append_column("source_row_number", pa.array(range(1, n + 1), pa.int32()))
           .append_column("file_sha256", pa.array([file_hash] * n, pa.string()))
           .append_column("ingested_at", pa.array([datetime.now(BKK)] * n, pa.timestamp("us", tz="Asia/Bangkok")))
    )

    # Guard 2 (only with a key): keep keys that are not in bronze yet, first copy inside the file
    if key is not None:
        con.register("incoming", batch)
        batch = con.sql(f'''
            SELECT *
            FROM incoming AS i
            WHERE NOT EXISTS (
                SELECT 1 FROM metroride.bronze.{table_name} AS b WHERE b.{key} = i.{key}
            )
            QUALIFY row_number() OVER (PARTITION BY {key} ORDER BY source_row_number) = 1
            ORDER BY source_row_number
        ''').to_arrow_table()
        con.unregister("incoming")

    if batch.num_rows == 0:
        print(f"SKIP {csv_path.name}: all {n} rows are already in bronze.{table_name}")
        return 0

    with catalog.begin_transaction() as txn:
        catalog.set_commit_message(f"bronze: ingest {csv_path.name}", author=AUTHOR)
        txn.load_table(("bronze", table_name)).append(batch)

    print(f"OK   {csv_path.name:28s} -> bronze.{table_name}: {n} rows read, "
          f"{batch.num_rows} appended, {n - batch.num_rows} skipped")
    return batch.num_rows


def ingest_month(month: str) -> None:
    # Load one month of source files (e.g. month="2026-06") into bronze.
    ingest_csv(SOURCE_DIR / f"trips_{month}.csv", "trips_raw", TRIP_COLUMNS, key="trip_id")
    ingest_csv(SOURCE_DIR / f"driver_events_{month}.csv", "driver_events_raw", EVENT_COLUMNS)

In [11]:
ingest_month("2026-06")

OK   trips_2026-06.csv            -> bronze.trips_raw: 132 rows read, 132 appended, 0 skipped


OK   driver_events_2026-06.csv    -> bronze.driver_events_raw: 397 rows read, 397 appended, 0 skipped


The June event file contains one `gps_ping` that the app delivered twice. Bronze keeps both copies:

In [12]:
sql('''
    SELECT event_id, count(*) AS copies_in_bronze, list(source_row_number) AS source_rows, any_value(source_file) AS source_file
    FROM metroride.bronze.driver_events_raw
    GROUP BY event_id
    HAVING count(*) > 1
''')

,event_id,copies_in_bronze,source_rows,source_file
0,BKK-20260601-M1-P,2,"[2, 3]",driver_events_2026-06.csv


## 4. Transform June data from bronze to silver

The silver tables are filled in dependency order:

1. `silver.trips`: new valid trips (the phase-1 transformation, unchanged).
2. `silver.driver_events`: new valid events. Events are matched to **validated** trips, so trips must go first.
3. `silver.trip_context`: trips joined with their arrival event.

### 4.1 `silver.trips`

The same SQL as in phase 1: pick bronze trips not yet in silver, parse the types, derive `service_date` and `time_slot`, and keep only rows that pass the validation rules.

In [13]:
TRIPS_SQL = '''
WITH new_rows AS (               -- 1. bronze rows that are not in silver yet
    SELECT b.*
    FROM metroride.bronze.trips_raw AS b
    WHERE NOT EXISTS (
        SELECT 1 FROM metroride.silver.trips AS s WHERE s.trip_id = b.trip_id
    )
),
typed AS (                       -- 2. parse text into proper types
    SELECT
        trim(trip_id)                         AS trip_id,
        trim(driver_id)                       AS driver_id,
        trim(service_zone)                    AS service_zone,
        lower(trim(status))                   AS status,
        TRY_CAST(requested_at AS TIMESTAMPTZ) AS requested_at,
        TRY_CAST(accepted_at  AS TIMESTAMPTZ) AS accepted_at,
        TRY_CAST(picked_up_at AS TIMESTAMPTZ) AS picked_up_at,
        TRY_CAST(completed_at AS TIMESTAMPTZ) AS completed_at,
        TRY_CAST(cancelled_at AS TIMESTAMPTZ) AS cancelled_at,
        TRY_CAST(fare_thb AS DECIMAL(10, 2))  AS fare_thb,
        ingested_at
    FROM new_rows
),
derived AS (                     --    derive date and slot in Bangkok local time
    SELECT
        *,
        CAST(timezone('Asia/Bangkok', requested_at) AS DATE) AS service_date,
        CASE hour(timezone('Asia/Bangkok', requested_at))
            WHEN 8  THEN 'morning'
            WHEN 18 THEN 'evening'
        END AS time_slot
    FROM typed
)
SELECT                           -- 3. validate; column order matches silver.trips
    trip_id, driver_id, service_zone, status,
    requested_at, accepted_at, picked_up_at, completed_at, cancelled_at,
    fare_thb, service_date, time_slot
FROM derived
WHERE trip_id IS NOT NULL
  AND driver_id IS NOT NULL
  AND service_zone IN ('Siam', 'Asok', 'Ari', 'Silom')
  AND requested_at IS NOT NULL
  AND accepted_at IS NOT NULL
  AND requested_at <= accepted_at
  AND time_slot IS NOT NULL
  AND (
        (status = 'completed'
         AND picked_up_at IS NOT NULL AND completed_at IS NOT NULL
         AND accepted_at <= picked_up_at AND picked_up_at <= completed_at
         AND fare_thb IS NOT NULL AND fare_thb >= 0
         AND cancelled_at IS NULL)
     OR (status = 'cancelled'
         AND cancelled_at IS NOT NULL AND accepted_at <= cancelled_at
         AND picked_up_at IS NULL AND completed_at IS NULL AND fare_thb IS NULL)
  )
QUALIFY row_number() OVER (PARTITION BY trip_id ORDER BY ingested_at) = 1   -- one row per trip_id
ORDER BY requested_at
'''


def transform_trips() -> int:
    # Append new, valid bronze trips to silver.trips.
    pending = sql('''
        SELECT count(*) AS n FROM metroride.bronze.trips_raw AS b
        WHERE NOT EXISTS (SELECT 1 FROM metroride.silver.trips AS s WHERE s.trip_id = b.trip_id)
    ''').n[0]
    clean = con.sql(TRIPS_SQL).to_arrow_table()
    if clean.num_rows == 0:
        print(f"silver.trips         : nothing to do ({pending} bronze rows not in silver)")
        return 0
    with catalog.begin_transaction() as txn:
        catalog.set_commit_message(f"silver: transform {clean.num_rows} new trips", author=AUTHOR)
        txn.load_table(("silver", "trips")).append(clean)
    print(f"silver.trips         : {pending} new bronze rows -> {clean.num_rows} appended, "
          f"{pending - clean.num_rows} rejected")
    return clean.num_rows

### 4.2 `silver.driver_events`

Each run looks only at bronze events whose `event_id` is not in silver yet, and gives each one a `check_result`:

| Situation | `check_result` | What happens |
|---|---|---|
| identical copies of the same event | – | collapsed to one row (`SELECT DISTINCT`) |
| same `event_id`, different content | `conflicting event_id` | rejected |
| missing id, unknown `event_type`, unreadable `event_at` | `missing id`, `unknown event_type`, `bad event_at` | rejected |
| latitude outside [-90, 90] or longitude outside [-180, 180] | `bad coordinates` | rejected |
| trip not in `silver.trips` yet | `waiting for trip` | not written; retried on the next run |
| `driver_id` differs from the trip's driver | `driver does not match trip` | rejected |
| everything fine | `ok` | appended to silver |

Rejected and waiting events stay in bronze, so nothing is lost. "Waiting" matters for a live stream: an event can arrive before the nightly trip file that it belongs to.

In [14]:
EVENT_TYPES = ("accepted", "gps_ping", "arrived_at_pickup", "cancelled")

EVENTS_SQL = f'''
WITH new_rows AS (               -- 1. bronze events that are not in silver yet
    SELECT b.*
    FROM metroride.bronze.driver_events_raw AS b
    WHERE NOT EXISTS (
        SELECT 1 FROM metroride.silver.driver_events AS s WHERE s.event_id = b.event_id
    )
),
typed AS (                       -- 2. parse text into proper types
    SELECT DISTINCT              --    identical repeated deliveries collapse into one row
        trim(event_id)                        AS event_id,
        trim(trip_id)                         AS trip_id,
        trim(driver_id)                       AS driver_id,
        lower(trim(event_type))               AS event_type,
        TRY_CAST(event_at AS TIMESTAMPTZ)     AS event_at,
        TRY_CAST(latitude  AS DECIMAL(9, 5))  AS latitude,
        TRY_CAST(longitude AS DECIMAL(9, 5))  AS longitude
    FROM new_rows
),
checked AS (                     -- 3. validate, and match each event to a validated trip
    SELECT
        e.*,
        CASE
            WHEN e.event_id IS NULL OR e.trip_id IS NULL OR e.driver_id IS NULL     THEN 'missing id'
            WHEN count(*) OVER (PARTITION BY e.event_id) > 1                        THEN 'conflicting event_id'
            WHEN e.event_type NOT IN {EVENT_TYPES}                                  THEN 'unknown event_type'
            WHEN e.event_at IS NULL                                                 THEN 'bad event_at'
            WHEN e.latitude IS NULL OR e.longitude IS NULL
              OR e.latitude NOT BETWEEN -90 AND 90
              OR e.longitude NOT BETWEEN -180 AND 180                               THEN 'bad coordinates'
            WHEN t.trip_id IS NULL                                                  THEN 'waiting for trip'
            WHEN t.driver_id <> e.driver_id                                         THEN 'driver does not match trip'
            ELSE 'ok'
        END AS check_result
    FROM typed AS e
    LEFT JOIN metroride.silver.trips AS t ON t.trip_id = e.trip_id
)
SELECT * FROM checked
'''


def transform_events() -> int:
    # Append new, valid bronze events to silver.driver_events (one row per event_id).
    received = sql('''
        SELECT count(*) AS n FROM metroride.bronze.driver_events_raw AS b
        WHERE NOT EXISTS (SELECT 1 FROM metroride.silver.driver_events AS s WHERE s.event_id = b.event_id)
    ''').n[0]
    checked = con.sql(EVENTS_SQL).to_arrow_table()
    con.register("checked_events", checked)
    results = dict(con.sql("SELECT check_result, count(*) FROM checked_events GROUP BY ALL").fetchall())
    clean = con.sql('''
        SELECT event_id, trip_id, driver_id, event_type, event_at, latitude, longitude
        FROM checked_events WHERE check_result = 'ok' ORDER BY event_at, event_id
    ''').to_arrow_table()
    con.unregister("checked_events")

    if clean.num_rows > 0:
        with catalog.begin_transaction() as txn:
            catalog.set_commit_message(f"silver: transform {clean.num_rows} new driver events", author=AUTHOR)
            txn.load_table(("silver", "driver_events")).append(clean)
    print(f"silver.driver_events : {received} new bronze rows -> {received - checked.num_rows} identical copies removed, "
          f"results {results}, {clean.num_rows} appended")
    return clean.num_rows

### 4.3 `silver.trip_context`

For every trip in `silver.trips` we look up its `arrived_at_pickup` event and compute both waits:

* `driver_arrived_at`: the arrival time, used only if the trip has **exactly one** arrival event and it lies **between acceptance and passenger pickup**. Otherwise it stays `NULL`.
* `driver_arrival_wait_minutes = driver_arrived_at − requested_at`
* `passenger_pickup_wait_minutes = picked_up_at − requested_at`

April–May trips have no app events and cancelled trips have no arrival, so their arrival columns are `NULL`.

**Compute, compare, upsert.** A trip's row can change after it was first written: for example, its arrival event arrives late from the app. So instead of only appending new trips, `write_changed_rows()`:

1. computes the whole table from silver with a query,
2. compares it with what is stored (`EXCEPT`) to find rows that are new or different, and
3. **upserts** only those rows by key. `table.upsert()` runs a DuckLake `MERGE`: matching keys are updated and new keys are inserted.

Running it again with no new data writes nothing, so it is idempotent. Recomputing the whole table is fine at this size; a production job would recompute only recent dates.

In [15]:
TRIP_CONTEXT_SQL = '''
WITH arrivals AS (               -- the arrival event(s) of each trip
    SELECT trip_id, min(event_at) AS arrived_at, count(*) AS arrival_events
    FROM metroride.silver.driver_events
    WHERE event_type = 'arrived_at_pickup'
    GROUP BY trip_id
),
joined AS (                      -- left join: every trip is kept
    SELECT
        t.*,
        CASE WHEN a.arrival_events = 1
              AND a.arrived_at BETWEEN t.accepted_at AND t.picked_up_at
             THEN a.arrived_at
        END AS driver_arrived_at
    FROM metroride.silver.trips AS t
    LEFT JOIN arrivals AS a ON a.trip_id = t.trip_id
)
SELECT                           -- column order matches silver.trip_context
    *,
    CAST(date_diff('second', requested_at, driver_arrived_at) / 60.0 AS DECIMAL(8, 2)) AS driver_arrival_wait_minutes,
    CAST(date_diff('second', requested_at, picked_up_at)      / 60.0 AS DECIMAL(8, 2)) AS passenger_pickup_wait_minutes
FROM joined
ORDER BY requested_at
'''


def write_changed_rows(table_id: tuple[str, str], query: str, keys: list[str]) -> None:
    # Compute a table with `query`, then upsert only the rows that are new or changed.
    # The query must return the table's columns in the table's order.
    name = ".".join(table_id)
    total = con.sql(f"SELECT count(*) FROM ({query})").fetchone()[0]
    changed = con.sql(f"SELECT * FROM ({query}) EXCEPT SELECT * FROM metroride.{name}").to_arrow_table()
    if changed.num_rows == 0:
        print(f"{name:21s}: {total} rows computed, nothing new or changed")
        return
    with catalog.begin_transaction() as txn:
        catalog.set_commit_message(f"{table_id[0]}: refresh {table_id[1]}", author=AUTHOR)
        result = txn.load_table(table_id).upsert(changed, join_cols=keys)
    print(f"{name:21s}: {total} rows computed -> {result.rows_inserted} inserted, "
          f"{result.rows_updated} updated, {total - changed.num_rows} unchanged")


def transform_bronze_to_silver() -> None:
    transform_trips()
    transform_events()
    write_changed_rows(("silver", "trip_context"), TRIP_CONTEXT_SQL, keys=["trip_id"])

Run all three for June. Notice that the first `trip_context` refresh also inserts the 269 April–May trips from phase 1.

In [16]:
transform_bronze_to_silver()

silver.trips         : 132 new bronze rows -> 132 appended, 0 rejected


silver.driver_events : 397 new bronze rows -> 1 identical copies removed, results {'ok': 396}, 396 appended


silver.trip_context  : 401 rows computed -> 401 inserted, 0 updated, 0 unchanged


Quick checks: `trip_context` has one row per trip, and every completed trip from June onwards found its arrival event.

In [17]:
sql('''
    SELECT count(*)                AS trip_context_rows,
           count(DISTINCT trip_id) AS distinct_trips,
           (SELECT count(*) FROM metroride.silver.trips) AS silver_trips_rows,
           count(*) FILTER (service_date >= DATE '2026-06-01' AND status = 'completed'
                            AND driver_arrived_at IS NULL) AS completed_from_june_without_arrival
    FROM metroride.silver.trip_context
''')

,trip_context_rows,distinct_trips,silver_trips_rows,completed_from_june_without_arrival
0,401,401,401,0


## 5. Transform June data from silver to gold

The gold query now reads `silver.trip_context` and returns all ten columns. The first six measures use the same formulas as in phase 1; the two new ones use the driver's arrival:

* `average_driver_arrival_wait_minutes`: average of `driver_arrival_wait_minutes` over completed trips.
* `arrived_within_8_minutes_pct`: `100 ×` completed trips with arrival wait ≤ 8 minutes `/` completed trips **with an arrival event**. `NULLIF(..., 0)` makes it `NULL` when no arrival events exist, as for April–May.

Rates and minutes are rounded only after aggregation. The same compute–compare–upsert function writes the result, so:

* June rows are **inserted**;
* April–May rows compute to exactly the values already stored (new columns `NULL`), so they are **left unchanged**;
* a late event that changes a trip would **update** that day's row.

In [18]:
GOLD_SQL = '''
SELECT
    service_date,
    time_slot,
    CAST(count(*) AS INTEGER)                                AS requests,
    CAST(count(*) FILTER (status = 'completed') AS INTEGER)  AS completed_trips,
    CAST(count(*) FILTER (status = 'cancelled') AS INTEGER)  AS cancelled_trips,
    CAST(round(100.0 * count(*) FILTER (status = 'cancelled') / count(*), 2) AS DECIMAL(5, 2))
                                                             AS cancellation_rate_pct,
    CAST(coalesce(sum(fare_thb) FILTER (status = 'completed'), 0) AS DECIMAL(12, 2))
                                                             AS gross_fare_thb,
    CAST(round(avg(passenger_pickup_wait_minutes) FILTER (status = 'completed'), 2) AS DECIMAL(8, 2))
                                                             AS average_passenger_pickup_wait_minutes,
    -- new in phase 2
    CAST(round(avg(driver_arrival_wait_minutes) FILTER (status = 'completed'), 2) AS DECIMAL(8, 2))
                                                             AS average_driver_arrival_wait_minutes,
    CAST(round(100.0
               * count(*) FILTER (status = 'completed' AND driver_arrival_wait_minutes <= 8)
               / NULLIF(count(*) FILTER (status = 'completed' AND driver_arrived_at IS NOT NULL), 0), 2)
         AS DECIMAL(5, 2))                                   AS arrived_within_8_minutes_pct
FROM metroride.silver.trip_context
GROUP BY service_date, time_slot
ORDER BY service_date, time_slot DESC
'''


def transform_silver_to_gold() -> None:
    write_changed_rows(("gold", "daily_service_performance"), GOLD_SQL, keys=["service_date", "time_slot"])


transform_silver_to_gold()

gold.daily_service_performance: 182 rows computed -> 60 inserted, 0 updated, 122 unchanged


In [19]:
gold.scan("service_date BETWEEN DATE '2026-05-31' AND DATE '2026-06-02'").to_pandas().sort_values(["service_date", "time_slot"], ascending=[True, False]).reset_index(drop=True)

,service_date,time_slot,requests,completed_trips,cancelled_trips,cancellation_rate_pct,gross_fare_thb,average_passenger_pickup_wait_minutes,average_driver_arrival_wait_minutes,arrived_within_8_minutes_pct
0,2026-05-31,morning,2,1,1,50.00,205.0,13.0,NaN,NaN
1,2026-05-31,evening,3,2,1,33.33,309.0,13.5,NaN,NaN
2,2026-06-01,morning,2,2,0,0.00,309.0,9.5,7.0,100.0
3,2026-06-01,evening,2,2,0,0.00,345.0,9.5,8.0,50.0
4,2026-06-02,morning,2,2,0,0.00,365.0,7.5,5.5,100.0
5,2026-06-02,evening,2,2,0,0.00,369.0,10.5,8.0,50.0


## 6. Ingest July data into the bronze layer

The same function as in Step 3. Each file creates one snapshot.

In [20]:
ingest_month("2026-07")

OK   trips_2026-07.csv            -> bronze.trips_raw: 136 rows read, 136 appended, 0 skipped
OK   driver_events_2026-07.csv    -> bronze.driver_events_raw: 409 rows read, 409 appended, 0 skipped


## 7. Transform July data from bronze to silver

Only bronze rows that are not in silver yet are processed; `trip_context` inserts the new July trips and leaves earlier trips unchanged.

In [21]:
transform_bronze_to_silver()

silver.trips         : 136 new bronze rows -> 136 appended, 0 rejected


silver.driver_events : 409 new bronze rows -> 1 identical copies removed, results {'ok': 408}, 408 appended
silver.trip_context  : 537 rows computed -> 136 inserted, 0 updated, 401 unchanged


## 8. Transform July data from silver to gold

Only the July date-and-slot rows are new, so only they are written.

In [22]:
transform_silver_to_gold()

gold.daily_service_performance: 244 rows computed -> 62 inserted, 0 updated, 182 unchanged


## 9. Ingest August data into the bronze layer

The same function as in Step 3. Each file creates one snapshot.

In [23]:
ingest_month("2026-08")

OK   trips_2026-08.csv            -> bronze.trips_raw: 136 rows read, 136 appended, 0 skipped
OK   driver_events_2026-08.csv    -> bronze.driver_events_raw: 409 rows read, 409 appended, 0 skipped


## 10. Transform August data from bronze to silver

Only bronze rows that are not in silver yet are processed; `trip_context` inserts the new August trips and leaves earlier trips unchanged.

In [24]:
transform_bronze_to_silver()

silver.trips         : 136 new bronze rows -> 136 appended, 0 rejected


silver.driver_events : 409 new bronze rows -> 1 identical copies removed, results {'ok': 408}, 408 appended


silver.trip_context  : 673 rows computed -> 136 inserted, 0 updated, 537 unchanged


## 11. Transform August data from silver to gold

Only the August date-and-slot rows are new, so only they are written.

In [25]:
transform_silver_to_gold()

gold.daily_service_performance: 306 rows computed -> 62 inserted, 0 updated, 244 unchanged


**Replay check.** If the scheduler re-runs the whole pipeline over every file, nothing changes: the file guard skips every file, and both transformations find nothing new.

In [26]:
for month in ["2026-06", "2026-07", "2026-08"]:
    ingest_month(month)
transform_bronze_to_silver()
transform_silver_to_gold()

SKIP trips_2026-06.csv: this file is already in bronze.trips_raw
SKIP driver_events_2026-06.csv: this file is already in bronze.driver_events_raw
SKIP trips_2026-07.csv: this file is already in bronze.trips_raw
SKIP driver_events_2026-07.csv: this file is already in bronze.driver_events_raw


SKIP trips_2026-08.csv: this file is already in bronze.trips_raw
SKIP driver_events_2026-08.csv: this file is already in bronze.driver_events_raw
silver.trips         : nothing to do (0 bronze rows not in silver)


silver.driver_events : 0 new bronze rows -> 0 identical copies removed, results {}, 0 appended


silver.trip_context  : 673 rows computed, nothing new or changed
gold.daily_service_performance: 306 rows computed, nothing new or changed


## 12. Display data in the bronze, silver and gold layers

### 12.1 Row counts per month and table

Things to notice:

* Bronze events have **one more row per month** than silver events: the repeated `gps_ping` delivery.
* `silver.trip_context` has exactly as many rows as `silver.trips`: the left join did not lose or duplicate any trip.
* Gold has one row per date and slot.

In [27]:
counts = sql('''
    SELECT 'bronze.trips_raw' AS table_name, substr(requested_at, 1, 7) AS month, count(*) AS rows
    FROM metroride.bronze.trips_raw GROUP BY ALL
    UNION ALL
    SELECT 'bronze.driver_events_raw', substr(event_at, 1, 7), count(*)
    FROM metroride.bronze.driver_events_raw GROUP BY ALL
    UNION ALL
    SELECT 'silver.trips', strftime(service_date, '%Y-%m'), count(*)
    FROM metroride.silver.trips GROUP BY ALL
    UNION ALL
    SELECT 'silver.driver_events', strftime(event_at, '%Y-%m'), count(*)
    FROM metroride.silver.driver_events GROUP BY ALL
    UNION ALL
    SELECT 'silver.trip_context', strftime(service_date, '%Y-%m'), count(*)
    FROM metroride.silver.trip_context GROUP BY ALL
    UNION ALL
    SELECT 'gold.daily_service_performance', strftime(service_date, '%Y-%m'), count(*)
    FROM metroride.gold.daily_service_performance GROUP BY ALL
''')
table_order = ["bronze.trips_raw", "bronze.driver_events_raw", "silver.trips", "silver.driver_events",
               "silver.trip_context", "gold.daily_service_performance"]
counts.pivot(index="month", columns="table_name", values="rows")[table_order].fillna(0).astype(int)

table_name,bronze.trips_raw,bronze.driver_events_raw,silver.trips,silver.driver_events,silver.trip_context,gold.daily_service_performance
month,,,,,,
2026-04,132,0,132,0,132,60
2026-05,137,0,137,0,137,62
2026-06,132,397,132,396,132,60
2026-07,136,409,136,408,136,62
2026-08,136,409,136,408,136,62


### 12.2 One trip through all three layers

We follow trip `BKK-20260601-M1`, the trip whose GPS ping was delivered twice.

**Bronze**: the raw trip row and the raw events, as received (all text, with ingest metadata). Note the two copies of the `-P` event:

In [28]:
TRIP = "BKK-20260601-M1"
bronze_trips.scan(f"trip_id = '{TRIP}'").to_pandas().T

,0
trip_id,BKK-20260601-M1
driver_id,D002
service_zone,Asok
requested_at,2026-06-01T08:03:00+07:00
accepted_at,2026-06-01T08:05:00+07:00
picked_up_at,2026-06-01T08:11:00+07:00
completed_at,2026-06-01T08:24:00+07:00
cancelled_at,None
status,completed
fare_thb,148.00


In [29]:
bronze_events.scan(f"trip_id = '{TRIP}'").select(
    "event_id", "event_type", "event_at", "latitude", "longitude", "source_file", "source_row_number"
).to_pandas()

,event_id,event_type,event_at,latitude,longitude,source_file,source_row_number
0,BKK-20260601-M1-A,accepted,2026-06-01T08:05:00+07:00,13.73600,100.52400,driver_events_2026-06.csv,1
1,BKK-20260601-M1-P,gps_ping,2026-06-01T08:06:00+07:00,13.73700,100.52500,driver_events_2026-06.csv,2
2,BKK-20260601-M1-P,gps_ping,2026-06-01T08:06:00+07:00,13.73700,100.52500,driver_events_2026-06.csv,3
3,BKK-20260601-M1-R,arrived_at_pickup,2026-06-01T08:09:00+07:00,13.73800,100.52600,driver_events_2026-06.csv,4


**Silver**: typed events with one row per `event_id`, and the trip's context row with both waits. The driver arrived 6 minutes after the request; the passenger was picked up 8 minutes after it.

In [30]:
silver_events.scan(f"trip_id = '{TRIP}'").to_pandas()

,event_id,trip_id,driver_id,event_type,event_at,latitude,longitude
0,BKK-20260601-M1-A,BKK-20260601-M1,D002,accepted,2026-06-01 08:05:00+07:00,13.736,100.524
1,BKK-20260601-M1-P,BKK-20260601-M1,D002,gps_ping,2026-06-01 08:06:00+07:00,13.737,100.525
2,BKK-20260601-M1-R,BKK-20260601-M1,D002,arrived_at_pickup,2026-06-01 08:09:00+07:00,13.738,100.526


In [31]:
trip_context.scan(f"trip_id = '{TRIP}'").to_pandas().T

,0
trip_id,BKK-20260601-M1
driver_id,D002
service_zone,Asok
status,completed
requested_at,2026-06-01 08:03:00+07:00
accepted_at,2026-06-01 08:05:00+07:00
picked_up_at,2026-06-01 08:11:00+07:00
completed_at,2026-06-01 08:24:00+07:00
cancelled_at,NaT
fare_thb,148.0


**Gold**: the date-and-slot row that this trip is counted in.

In [32]:
gold.scan("service_date = DATE '2026-06-01' AND time_slot = 'morning'").to_pandas().T

,0
service_date,2026-06-01 00:00:00
time_slot,morning
requests,2
completed_trips,2
cancelled_trips,0
cancellation_rate_pct,0.0
gross_fare_thb,309.0
average_passenger_pickup_wait_minutes,9.5
average_driver_arrival_wait_minutes,7.0
arrived_within_8_minutes_pct,100.0


### 12.3 The gold table by month

Counts are summed first and the cancellation rate is calculated from the sums, never by averaging daily rates. `slots_with_arrival_data` shows that the driver-arrival measures exist only from June onwards.

In [33]:
sql('''
    SELECT strftime(service_date, '%Y-%m')                             AS month,
           count(*)                                                    AS slots,
           CAST(sum(requests) AS INTEGER)                              AS requests,
           CAST(sum(completed_trips) AS INTEGER)                       AS completed,
           CAST(sum(cancelled_trips) AS INTEGER)                       AS cancelled,
           round(100.0 * sum(cancelled_trips) / sum(requests), 2)      AS cancellation_rate_pct,
           sum(gross_fare_thb)                                         AS gross_fare_thb,
           count(average_driver_arrival_wait_minutes)                  AS slots_with_arrival_data
    FROM metroride.gold.daily_service_performance
    GROUP BY ALL
    ORDER BY month
''')

,month,slots,requests,completed,cancelled,cancellation_rate_pct,gross_fare_thb,slots_with_arrival_data
0,2026-04,60,132,124,8,6.06,22575.0,0
1,2026-05,62,137,127,10,7.30,23089.0,0
2,2026-06,60,132,124,8,6.06,22616.0,60
3,2026-07,62,136,127,9,6.62,23104.0,62
4,2026-08,62,136,126,10,7.35,22825.0,62


### 12.4 Snapshots written in phase 2

Every committed change is a snapshot. Below are the ones this notebook created: table creation, the gold `ALTER`, and one snapshot per ingested file and per transformation. Runs that found nothing to do created no snapshot.

In [34]:
snapshots_df().query("snapshot_id > @PHASE1_LAST_SNAPSHOT")

,snapshot_id,snapshot_time,schema_version,changes,author,commit_message
9,9,2026-10-15 21:51:42.408534+07:00,5,"[(tables_created, [bronze.driver_events_raw])]",NaN,NaN
10,10,2026-10-15 21:51:42.501837+07:00,6,"[(tables_created, [silver.driver_events])]",NaN,NaN
11,11,2026-10-15 21:51:42.551135+07:00,7,"[(tables_created, [silver.trip_context])]",NaN,NaN
12,12,2026-10-15 21:51:42.620144+07:00,8,"[(schemas_created, [gold])]",NaN,NaN
13,13,2026-10-15 21:51:42.651312+07:00,9,"[(tables_created, [gold.daily_service_performance])]",NaN,NaN
14,14,2026-10-15 21:51:42.709174+07:00,9,"[(tables_inserted_into, [9])]",phase-2 tutorial,gold: build April-May with the phase-1 columns
15,15,2026-10-15 21:51:42.888241+07:00,10,"[(tables_altered, [9])]",phase-2 tutorial,gold: add driver-arrival measures
16,16,2026-10-15 21:51:43.095897+07:00,10,"[(tables_inserted_into, [3])]",phase-2 tutorial,bronze: ingest trips_2026-06.csv
17,17,2026-10-15 21:51:43.193121+07:00,10,"[(tables_inserted_into, [5])]",phase-2 tutorial,bronze: ingest driver_events_2026-06.csv
18,18,2026-10-15 21:51:43.323224+07:00,10,"[(tables_inserted_into, [4])]",phase-2 tutorial,silver: transform 132 new trips


## 13. Schema evolution: the evolved gold table

`gold.daily_service_performance` was created with 8 columns, filled with April–May, and then got 2 more columns. Let us look at what DuckLake recorded.

### 13.1 Column history in the catalog

Each column in `ducklake_column` has a `begin_snapshot` (when it was added) and an `end_snapshot` (when it was dropped, `NULL` while it exists). The two new columns start at the `ALTER` snapshot.

In [35]:
sql(f'''
    SELECT c.column_id, c.column_name, c.column_type, c.nulls_allowed,
           c.begin_snapshot AS added_in_snapshot,
           c.end_snapshot   AS dropped_in_snapshot
    FROM __ducklake_metadata_metroride.ducklake_column AS c
    JOIN __ducklake_metadata_metroride.ducklake_table  AS t USING (table_id)
    JOIN __ducklake_metadata_metroride.ducklake_schema AS s USING (schema_id)
    WHERE s.schema_name = 'gold' AND t.table_name = 'daily_service_performance'
    ORDER BY c.column_id
''')

,column_id,column_name,column_type,nulls_allowed,added_in_snapshot,dropped_in_snapshot
0,1,service_date,date,False,13,<NA>
1,2,time_slot,varchar,False,13,<NA>
2,3,requests,int32,False,13,<NA>
3,4,completed_trips,int32,False,13,<NA>
4,5,cancelled_trips,int32,False,13,<NA>
5,6,cancellation_rate_pct,"decimal(5,2)",False,13,<NA>
6,7,gross_fare_thb,"decimal(12,2)",False,13,<NA>
7,8,average_passenger_pickup_wait_minutes,"decimal(8,2)",True,13,<NA>
8,9,average_driver_arrival_wait_minutes,"decimal(8,2)",True,15,<NA>
9,10,arrived_within_8_minutes_pct,"decimal(5,2)",True,15,<NA>


The snapshots that touched the gold table: created, filled with April–May, altered, then one upsert per month. Note `schema_version`: it increases with the `ALTER`.

In [36]:
gold_table_id = str(int(sql('''
    SELECT t.table_id
    FROM __ducklake_metadata_metroride.ducklake_table AS t
    JOIN __ducklake_metadata_metroride.ducklake_schema AS s USING (schema_id)
    WHERE s.schema_name = 'gold' AND t.table_name = 'daily_service_performance' AND t.end_snapshot IS NULL
''').table_id[0]))


def touches_gold(changes) -> bool:
    values = [v for _, vals in changes for v in vals]
    return gold_table_id in values or "gold.daily_service_performance" in values


snaps = snapshots_df()
snaps[snaps.changes.apply(touches_gold)]

,snapshot_id,snapshot_time,schema_version,changes,author,commit_message
13,13,2026-10-15 21:51:42.651312+07:00,9,"[(tables_created, [gold.daily_service_performance])]",NaN,NaN
14,14,2026-10-15 21:51:42.709174+07:00,9,"[(tables_inserted_into, [9])]",phase-2 tutorial,gold: build April-May with the phase-1 columns
15,15,2026-10-15 21:51:42.888241+07:00,10,"[(tables_altered, [9])]",phase-2 tutorial,gold: add driver-arrival measures
21,21,2026-10-15 21:51:43.663708+07:00,10,"[(tables_inserted_into, [9])]",phase-2 tutorial,gold: refresh daily_service_performance
27,27,2026-10-15 21:51:44.215394+07:00,10,"[(tables_inserted_into, [9])]",phase-2 tutorial,gold: refresh daily_service_performance
33,33,2026-10-15 21:51:44.784943+07:00,10,"[(tables_inserted_into, [9])]",phase-2 tutorial,gold: refresh daily_service_performance


### 13.2 Time travel across the schema change

Reading the table **as of a snapshot** also uses the schema **of that snapshot**. Before the `ALTER` the table has 8 columns; right after it, 10 columns with the new ones empty; now, 10 columns with data from June onwards.

In [37]:
versions = {
    f"snapshot {gold_snapshot_before_alter} (before ALTER)": gold.scan().with_snapshot(gold_snapshot_before_alter),
    f"snapshot {gold_snapshot_after_alter} (after ALTER)":  gold.scan().with_snapshot(gold_snapshot_after_alter),
    "current": gold.scan(),
}
pd.DataFrame(
    [(label, len(df := scan.to_pandas()), len(df.columns),
      int(df["average_driver_arrival_wait_minutes"].notna().sum()) if "average_driver_arrival_wait_minutes" in df else None)
     for label, scan in versions.items()],
    columns=["version", "rows", "columns", "rows_with_arrival_measure"],
)

,version,rows,columns,rows_with_arrival_measure
0,snapshot 14 (before ALTER),122,8,NaN
1,snapshot 15 (after ALTER),122,10,0.0
2,current,306,10,184.0


The same row, `2026-05-31 evening`, before and after the change:

In [38]:
KEY = "service_date = DATE '2026-05-31' AND time_slot = 'evening'"
print(f"Before ALTER (snapshot {gold_snapshot_before_alter}):")
display(gold.scan(KEY).with_snapshot(gold_snapshot_before_alter).to_pandas())
print("Now:")
display(gold.scan(KEY).to_pandas())

Before ALTER (snapshot 14):


,service_date,time_slot,requests,completed_trips,cancelled_trips,cancellation_rate_pct,gross_fare_thb,average_passenger_pickup_wait_minutes
0,2026-05-31,evening,3,2,1,33.33,309.0,13.5


Now:


,service_date,time_slot,requests,completed_trips,cancelled_trips,cancellation_rate_pct,gross_fare_thb,average_passenger_pickup_wait_minutes,average_driver_arrival_wait_minutes,arrived_within_8_minutes_pct
0,2026-05-31,evening,3,2,1,33.33,309.0,13.5,NaN,NaN


The same comparison in SQL with `AT (VERSION => ...)`: `SELECT *` returns a different set of columns depending on the version.

In [39]:
print(sql(f"SELECT * FROM metroride.gold.daily_service_performance AT (VERSION => {gold_snapshot_before_alter}) LIMIT 0").columns.tolist())
print(sql("SELECT * FROM metroride.gold.daily_service_performance LIMIT 0").columns.tolist())

['service_date', 'time_slot', 'requests', 'completed_trips', 'cancelled_trips', 'cancellation_rate_pct', 'gross_fare_thb', 'average_passenger_pickup_wait_minutes']
['service_date', 'time_slot', 'requests', 'completed_trips', 'cancelled_trips', 'cancellation_rate_pct', 'gross_fare_thb', 'average_passenger_pickup_wait_minutes', 'average_driver_arrival_wait_minutes', 'arrived_within_8_minutes_pct']


### 13.3 Old data files were not rewritten

The Parquet file that holds April–May was written **before** the `ALTER`, so it physically contains only the 8 original columns. Files written later contain all 10. When DuckLake reads an old file it fills the missing columns with `NULL`; that is why adding a column is cheap, even on a very large table. (Files written by `upsert` also carry an internal row-id column, which we hide here.)

In [40]:
files = sql(f'''
    SELECT f.path, f.record_count, f.begin_snapshot AS added_in_snapshot
    FROM __ducklake_metadata_metroride.ducklake_data_file AS f
    WHERE f.table_id = {gold_table_id} AND f.end_snapshot IS NULL
    ORDER BY f.begin_snapshot
''')
gold_dir = LAKE_DIR / "data" / "gold" / "daily_service_performance"
files["columns_in_file"] = [
    con.sql(f'''
        SELECT count(*) FROM parquet_schema('{gold_dir / p}')
        WHERE num_children IS NULL AND name NOT LIKE '%ducklake_internal%'
    ''').fetchone()[0]
    for p in files.path
]
files["months"] = [
    ", ".join(sorted(con.sql(f"SELECT DISTINCT strftime(service_date, '%Y-%m') FROM read_parquet('{gold_dir / p}')").df().iloc[:, 0]))
    for p in files.path
]
files

,path,record_count,added_in_snapshot,columns_in_file,months
0,ducklake-01a1400c-2f24-7d9e-93a9-38b55399f571.parquet,122,14,8,"2026-04, 2026-05"
1,ducklake-01a1400c-32d7-741a-8ec5-ad513f2d4271.parquet,60,21,10,2026-06
2,ducklake-01a1400c-34f4-7fdc-8df7-10d14fe940ce.parquet,62,27,10,2026-07
3,ducklake-01a1400c-3732-7094-a329-974cf77226ff.parquet,62,33,10,2026-08


### 13.4 Old rows next to new rows

The last day of May (written with the phase-1 schema) next to the first day of June (written with the phase-2 schema):

In [41]:
gold.scan("service_date IN (DATE '2026-05-31', DATE '2026-06-01')").to_pandas().sort_values(["service_date", "time_slot"], ascending=[True, False]).reset_index(drop=True)

,service_date,time_slot,requests,completed_trips,cancelled_trips,cancellation_rate_pct,gross_fare_thb,average_passenger_pickup_wait_minutes,average_driver_arrival_wait_minutes,arrived_within_8_minutes_pct
0,2026-05-31,morning,2,1,1,50.00,205.0,13.0,NaN,NaN
1,2026-05-31,evening,3,2,1,33.33,309.0,13.5,NaN,NaN
2,2026-06-01,morning,2,2,0,0.00,309.0,9.5,7.0,100.0
3,2026-06-01,evening,2,2,0,0.00,345.0,9.5,8.0,50.0


## Summary

* **Starting point.** We copied the phase-1 lakehouse and pointed its catalog at the copied data folder (`data_path` in `ducklake_metadata`).
* **New tables.** `bronze.driver_events_raw` keeps every received event as text; `silver.driver_events` keeps one typed, valid row per `event_id`; `silver.trip_context` joins every trip to its arrival event.
* **Altered table.** `gold.daily_service_performance` gained two nullable columns through `update_schema()`. The change is metadata-only: old Parquet files are not rewritten, old rows read `NULL`, and time travel shows each snapshot with its own schema.
* **Monthly runs.** Each month is ingest → bronze-to-silver → silver-to-gold. Append-only tables (`silver.trips`, `silver.driver_events`) take only new keys; derived tables (`silver.trip_context`, gold) are recomputed, compared, and **upserted**, so late or corrected data updates the right rows and replays change nothing.

**Try it yourself**

1. Make an event CSV with one bad row (for example `latitude` = `123.0`) and one event for a September trip. Ingest it and run `transform_events()`. Which `check_result` does each row get, and which rows reach silver?
2. Simulate a **late event**: delete the arrival event of one June trip from `silver.driver_events` (`silver_events.delete("event_id = '...'")`), refresh `trip_context` and gold, then put the event back and refresh again. Which gold row is updated each time?
3. Use time travel to compare `gold.daily_service_performance` before and after the July run.
4. Phase 3 adds a nullable `is_rainy` column to the gold **grouping key**. Why is that a bigger change than the two columns we added here?